# Bölüm klipleri: netleştirme (ücretsiz ekran kartında 2 kat büyütme)

`klip-uret` 704x416 boyutunda klip üretir; bölümde 1280x720'ye büyütülünce bulanık görünür.
Bu not defteri her kareyi **Real-ESRGAN x2** ile 1408x832'ye büyütüp keskinleştirir.

**Ayarlar:** Accelerator **GPU T4**, Internet **On**, sağ üstte **Add Input** →
`klip-uret` not defterinin çıktısı (ya da `klipler-*.zip` dataset'i). Sonra **Run All**.
Bitince Output'tan `klipler-net-*.zip` dosyasını indirin.

In [ ]:
# 1) Kurulum
!pip install -q spandrel imageio imageio-ffmpeg

In [ ]:
# 2) Klipleri bul ve modeli yükle
import glob, os, time, shutil, zipfile, urllib.request
import numpy as np, torch, imageio.v2 as imageio
from spandrel import ModelLoader

IN_DIR = "/kaggle/working/girdi"
OUT_DIR = "/kaggle/working/clips"
os.makedirs(IN_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

# Girdi: klip-uret çıktısındaki clips/*.mp4 ya da klipler-*.zip
for z in glob.glob("/kaggle/input/**/klipler-*.zip", recursive=True):
    zipfile.ZipFile(z).extractall(IN_DIR)
for f in glob.glob("/kaggle/input/**/c[0-9]*.mp4", recursive=True):
    shutil.copy(f, IN_DIR)
clips = sorted(glob.glob(f"{IN_DIR}/**/*.mp4", recursive=True))
assert clips, "Klip bulunamadı. 'Add Input' ile klip-uret çıktısını ekleyin."
print(len(clips), "klip bulundu")

W = "/tmp/RealESRGAN_x2plus.pth"  # Output'a düşmesin, yanlışlıkla indirilmesin
if not os.path.exists(W):
    urllib.request.urlretrieve(
        "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth", W)
model = ModelLoader().load_from_file(W).model.cuda().eval().half()
BATCH = 4

In [ ]:
# 3) Her klibi 2 kat büyüt
@torch.inference_mode()
def upscale(frames):
    x = torch.from_numpy(np.stack(frames)).cuda().permute(0, 3, 1, 2).half() / 255
    y = model(x).clamp(0, 1).mul(255).round().byte().permute(0, 2, 3, 1).cpu().numpy()
    return list(y)

for src in clips:
    key = os.path.splitext(os.path.basename(src))[0]
    out = os.path.join(OUT_DIR, f"{key}.mp4")
    if os.path.exists(out):
        print(f"✔ {key} zaten var"); continue
    t0 = time.time()
    r = imageio.get_reader(src)
    fps = r.get_meta_data().get("fps", 24)
    w = imageio.get_writer(out, fps=fps, codec="libx264", quality=9, macro_block_size=16)
    buf = []
    for fr in r:
        buf.append(fr[:, :, :3])
        if len(buf) == BATCH:
            for o in upscale(buf): w.append_data(o)
            buf = []
    if buf:
        for o in upscale(buf): w.append_data(o)
    w.close(); r.close()
    torch.cuda.empty_cache()
    print(f"✔ {key}  {time.time() - t0:.0f} sn")

In [ ]:
# 4) Zip'le
import json
plans = glob.glob("/kaggle/input/**/plan.json", recursive=True)
name = "neo-genesis-2099-e1"
if plans:
    p = json.load(open(plans[0])); name = f"{p['series']}-e{p['episode']}"
zip_base = f"/kaggle/working/klipler-net-{name}"
shutil.make_archive(zip_base, "zip", OUT_DIR)
shutil.rmtree(IN_DIR, ignore_errors=True)
print("🎬 İndirilecek dosya:", zip_base + ".zip")